# ARC2 014 V 27B service probe (v2: python3.12 + L4)
Single vLLM server (TP4/64K/C4/FP8, KV default, no MTP) under /usr/bin/python3.12, hello xhigh/low short protocol, then P2 if budget remains. Diagnostic only.

In [ ]:
import base64, hashlib, json, subprocess, sys, time
from pathlib import Path
notebook_started = time.time()
source_dir = Path('/kaggle/working/014-v-probe-code')
source_dir.mkdir(exist_ok=False)
blobs = json.loads(base64.b64decode('...'))
expected = {
  "vdriver.py": "0aa6675b6eb8c9fff17145bac13f7308c19cc85323ca54a26f4c9b1112279ba3",
  "run_reasoning_probe.py": "827934ec2865102eea31493acc2bd230da2fa1ebc2315cfeaf558fdde561e1c9",
  "p2-prompts.json": "9f0c23dc53994fc21a937ebfc1ea51d11af4377281ed07fa44bdcb723854dd83"
}
for name, encoded in blobs.items():
    raw = base64.b64decode(encoded)
    assert hashlib.sha256(raw).hexdigest() == expected[name], name
    (source_dir / name).write_bytes(raw)
PY312 = '/usr/bin/python3.12'
assert Path(PY312).is_file(), 'python3.12 binary missing (see env-probe evidence)'
ver = subprocess.run([PY312, '--version'], capture_output=True, text=True)
print('PY312', (ver.stdout + ver.stderr).strip(), flush=True)
assert 'Python 3.12' in (ver.stdout + ver.stderr), 'not python 3.12'
print(json.dumps({'event': 'boot', 'source_sha256': expected, 'py312': (ver.stdout + ver.stderr).strip(), 'python': sys.version, 'notebook_started': notebook_started}), flush=True)
result = subprocess.run([PY312, '-u', str(source_dir / 'vdriver.py'), '--work', '/kaggle/working/vprobe', '--prompts', str(source_dir / 'p2-prompts.json'), '--session-seconds', '3300', '--pip-python', sys.executable], cwd=str(source_dir))
print('VDRIVER_RETURN_CODE', result.returncode, flush=True)
assert result.returncode == 0, 'Inspect /kaggle/working/vprobe/run.json'
